In [ ]:
import plotly.graph_objects as go

def twh_to_ej(twh):
    """
    Convert Terawatt-hours (TWh) to Exajoules (EJ).

    Parameters:
    twh (float): Energy in Terawatt-hours.

    Returns:
    float: Energy in Exajoules.
    """
    conversion_factor = 0.0036
    return twh * conversion_factor

def xy(d):
    return zip(*sorted(d.items())) if d else ([], [])

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/coal_ammonia_blend_const_value_cp.xml`
    * `/input/policy/korea-2035/power/coal_ammonia_blend_const_techs.xml`
    * `/input/policy/korea-2035/power/gas_H2_blend_const_value.xml`
    * `/input/policy/korea-2035/power/gas_H2_blend_const_techs.xml`

# Co-firing

The 11th BPESD projects total hydrogen + ammonia power generation at 15.5 TWh in 2030, 32.8 TWh in 2035, and 43.9 TWh in 2038. However, it does not specify the individual generation amounts for hydrogen and ammonia. In contrast, the 10th BPESD projected hydrogen power generation of 6.1 TWh and ammonia power generation of 6.9 TWh in 2030.

In this study, it is assumed that the annual ratio of hydrogen to ammonia generation in the 11th Basic Plan is the same as the ratio in 2030 under the 10th Basic Plan. Accordingly, the hydrogen generation share is $\frac{6.1}{13} = 46.9\%$, and the ammonia generation share is $53.1\%$.

Coal power generations (TWh) are projected for 2023, 2030, and 2035 in BPESD. The generation for 2025 is calculated by linear interpolation. These generations are prjected as the figure below.

In [ ]:
dictCapTWhH2 = {2020: 0, 2023: 0, 2030: 15.5*0.469, 2035: 32.8*0.469}
dictCapTWhH2[2025] = dictCapTWhH2[2023] + (dictCapTWhH2[2030] - dictCapTWhH2[2023]) * (2/7)

In [5]:
dictCapTWhAmmonia = {2020: 0, 2023: 0, 2030: 15.5*(1-0.469), 2035: 32.8*(1-0.469)}
dictCapTWhAmmonia[2025] = dictCapTWhAmmonia[2023] + (dictCapTWhAmmonia[2030] - dictCapTWhAmmonia[2023]) * (2/7)

In [6]:
years_cap, values_cap = xy(dictCapTWhH2)
years_alt, values_alt = xy(dictCapTWhAmmonia)

fig = go.Figure()

for name, x, y, dash in [
    ("Hydrogen co-firing (LNG)", years_cap, values_cap, None),
    ("Ammonia co-firing (Coal)", years_alt, values_alt, "dash"),
]:
    fig.add_trace(go.Scatter(
        x=list(x), y=list(y),
        mode='lines+markers',
        name=name,
        line=(dict(dash=dash) if dash else None)
    ))

# Build annotations without repeating blocks
target_years = [2023, 2030, 2035]
annotations = []
for d in (dictCapTWhH2, dictCapTWhAmmonia):
    for yr in target_years:
        val = d.get(yr)
        if val is not None:
            annotations.append(go.layout.Annotation(
                x=yr, y=val,
                xanchor='center', yanchor='bottom',
                text=f"{val:.1f} TWh",
                showarrow=True, arrowhead=1, ax=0, ay=-20
            ))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=800, height=600,
    annotations=annotations,
    xaxis=dict(title='Year', title_font=dict(size=18), tickfont=dict(size=15)),
    yaxis=dict(title='TWh',  title_font=dict(size=18), tickfont=dict(size=15)),
)

fig.write_image("../figures/co-firing.png", scale=2)
fig.show()